<h3>문자, 재귀, 토큰 청킹</h3>
<p><strong>개념</strong> 청킹은 긴 문서를 검색 단위로 나누는 작업입니다. 같은 크기 숫자라도 문자를 세는지 토큰을 세는지에 따라 결과가 달라집니다.</p>

In [1]:
%pip install -q "langchain-text-splitters==1.1.2" "tiktoken==0.14.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = "배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니다.\n\n환불은 수령 후 7일 이내 신청합니다."


<h3>같은 문서를 세 기준으로 나누기</h3>
<p><strong>확인할 점</strong> 문자 방식은 고정 폭과 중복을 직접 적용합니다. 재귀 방식은 빈 줄과 문장 경계를 먼저 사용하고, 토큰 방식은 모델 입력 한도를 관리합니다.</p>

In [3]:
def character_chunks(value: str, size: int = 35, overlap: int = 8) -> list[str]:
    step = size - overlap
    return [value[i:i + size] for i in range(0, len(value), step)]
def overlap_length(left: str, right: str) -> int:
    limit = min(len(left), len(right))
    return max((n for n in range(1, limit + 1) if left[-n:] == right[:n]), default=0)
character = character_chunks(text)
print("character:", character)
print("character overlaps:", [overlap_length(a, b) for a, b in zip(character, character[1:])])


character: ['배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니', '틀이 더 걸립니다.\n\n환불은 수령 후 7일 이내 신청합니다.', '신청합니다.']
character overlaps: [8, 6]


In [4]:
recursive = RecursiveCharacterTextSplitter(
    chunk_size=35, chunk_overlap=8, keep_separator="end",
    separators=["\n\n", ". ", " ", ""]
).split_text(text)
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=20, chunk_overlap=4
)
token_chunks = token_splitter.split_text(text)
print("recursive:", recursive)
print("recursive overlaps:", [overlap_length(a, b) for a, b in zip(recursive, recursive[1:])])
print("token chunks:", token_chunks)


recursive: ['배송은 결제 다음 날 시작합니다.', '제주 지역은 이틀이 더 걸립니다.', '환불은 수령 후 7일 이내 신청합니다.']
recursive overlaps: [0, 0]
token chunks: ['배송은 결제 다음 날 시작합니다. 제주 지역은', '지역은 이틀이 더 걸립니다.', '환불은 수령 후 7일 이내 신청합니다.']


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 오버랩이 청크 크기 이상이면 다음 시작점이 앞으로 나아가지 않습니다. 설정 오류를 실행 전에 차단합니다.</p>

In [5]:
try:
    chunk_size, overlap = 20, 20
    assert 0 <= overlap < chunk_size, "overlap은 chunk_size보다 작아야 합니다"
except AssertionError as error:
    print(f"caught: {error}")


caught: overlap은 chunk_size보다 작아야 합니다
